# Module 8 lab · Statistics and estimation

The code from every lesson of [this module](https://aicourses.dev/mathematics/statistics), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 8.2 Correlation, and why it is not causation

Correlation is one number for moving together, and this lesson shows what it misses and how a pattern can reverse when you split the groups. It teaches you to distrust a number you have not looked behind.

[Read the lesson](https://aicourses.dev/mathematics/statistics/correlation)

### One number for moving together

For the five points, $\bar{x} = 3$ and $\bar{y} = 3$. The centred $x$ values are −2, −1, 0, 1, 2 and the centred $y$ values are −1, −2, 1, 0, 2. The products are 2, 2, 0, 0, 4, which add to 8. The squares of the centred $x$ add to 4 + 1 + 0 + 1 + 4 = 10 and those of $y$ to 1 + 4 + 1 + 0 + 4 = 10. So $r = 8 / \sqrt{10 \times 10} = 0.8$.

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 1, 4, 3, 5], dtype=float)
top = bottom_x = bottom_y = 0.0
for xi, yi in zip(x, y):
    top += (xi - x.mean()) * (yi - y.mean())
    bottom_x += (xi - x.mean()) ** 2
    bottom_y += (yi - y.mean()) ** 2
print(top, bottom_x, bottom_y, round(top / np.sqrt(bottom_x * bottom_y), 3))
print(round(np.corrcoef(x, y)[0, 1], 3))

*The lesson shows:*

```text
8.0 10.0 10.0 0.8
0.8
```

### The penguins

Now the twelve penguins, flipper against mass, then all 342.

In [ ]:
flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)  # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
print(round(np.corrcoef(flipper, mass)[0, 1], 3))

import urllib.request
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "penguins.csv", "penguins.csv")
F, M = np.loadtxt("penguins.csv", delimiter=",", skiprows=1,
                  usecols=(1, 2), unpack=True)  # the 342 birds
print(len(F), round(np.corrcoef(F, M)[0, 1], 3))

*The lesson shows:*

```text
0.815
342 0.871
```

### What one number misses

First, it sees only straight-line links. The values $x = -2, -1, 0, 1, 2$ and their squares are as tightly linked as numbers can be, since one is computed from the other, yet:

In [ ]:
t = np.array([-2, -1, 0, 1, 2], dtype=float)
print(round(np.corrcoef(t, t ** 2)[0, 1], 3))

*The lesson shows:*

```text
0.0
```

Second, it can hide groups. In the penguin table, the three species are three groups, and the correlation inside each one is weaker than for all of them together:

In [ ]:
sp = np.loadtxt("penguins.csv", delimiter=",", skiprows=1,
                usecols=0, dtype=str)
for s in ["Adelie", "Chinstrap", "Gentoo"]:
    k = sp == s
    print(s, k.sum(), round(np.corrcoef(F[k], M[k])[0, 1], 3))

*The lesson shows:*

```text
Adelie 151 0.468
Chinstrap 68 0.642
Gentoo 123 0.703
```

In [ ]:
men = np.array([[80, 56], [20, 4]])      # applied, admitted: X, Y
women = np.array([[20, 16], [80, 20]])  # made-up counts
print(men[:, 1] / men[:, 0], women[:, 1] / women[:, 0])
print(men[:, 1].sum() / men[:, 0].sum(),
      women[:, 1].sum() / women[:, 0].sum())

*The lesson shows:*

```text
[0.7 0.2] [0.8  0.25]
0.6 0.36
```

## 8.3 Likelihood

The likelihood says how well a setting of the knobs explains what was seen, and taking logs turns a product into a sum. It leads straight to the next lesson and returns in Module 09 as the loss.

[Read the lesson](https://aicourses.dev/mathematics/statistics/likelihood)

### Take logs

Each of the 7 heads contributes $\ln p$ and each of the 3 tails $\ln(1-p)$, added instead of multiplied. At $p = 0.7$: $\ln 0.7 = -0.3567$ and $\ln 0.3 = -1.2040$, so $7 \times (-0.3567) + 3 \times (-1.2040) = -2.4967 - 3.6119 = -6.109$. At $p = 0.5$: $10 \times \ln 0.5 = -6.931$. At $p = 0.9$: $7 \times (-0.1054) + 3 \times (-2.3026) = -7.645$. The log of a bigger number is bigger, so the order is the same: 0.7 is best, and its log-likelihood is the highest.

In [ ]:
import numpy as np

heads, tails = 7, 3
for p in [0.5, 0.7, 0.9]:
    lik = p ** heads * (1 - p) ** tails
    print(p, round(lik, 6), round(np.log(lik), 3))
grid = np.linspace(0.01, 0.99, 99)
loglik = heads * np.log(grid) + tails * np.log(1 - grid)
print(round(grid[loglik.argmax()], 2), round(loglik.max(), 3))

*The lesson shows:*

```text
0.5 0.000977 -6.931
0.7 0.002224 -6.109
0.9 0.000478 -7.645
0.7 -6.109
```

The loop works out each setting’s likelihood and its log. The last two lines do 99 settings at once: `grid` holds chances from 0.01 to 0.99, `loglik` the log-likelihood at each, and `argmax` gives the position of the biggest. The best setting on the grid is 0.7. Now the case where multiplying fails: 2,000 flips with 1,400 heads, at $p = 0.7$.

In [ ]:
n_h, n_t = 1400, 600
print(0.7 ** n_h * 0.3 ** n_t)
print(round(n_h * np.log(0.7) + n_t * np.log(0.3), 1))

*The lesson shows:*

```text
0.0
-1221.7
```

## 8.4 Maximum likelihood

Pick the setting that explains the data best, and for a line with bell-shaped noise that turns out to be least squares. It shows that the error you chose in Module 00 had a reason behind it.

[Read the lesson](https://aicourses.dev/mathematics/statistics/maximum-likelihood)

### A bell: the average

For the three measurements 197, 198 and 205, the average is 600 / 3 = 200, and the squared distances from it are 9, 4 and 25, which add to 38. From the middle value 198 they are 1, 0 and 49, which add to 50. The average has the smaller sum, so under bell-shaped noise it has the higher likelihood.

In [ ]:
import numpy as np

y = np.array([197, 198, 205], dtype=float)     # one flipper, in mm
best = None
for mu in np.arange(190, 210.5, 0.5):
    ss = ((y - mu) ** 2).sum()
    if best is None or ss < best[1]:
        best = (float(mu), float(ss))
print(best, y.mean())

*The lesson shows:*

```text
(200.0, 38.0) 200.0
```

### A line with bell-shaped noise

The code sets the spread to the root of the best line’s mean squared error (it can be shown that this is the best spread; the last line of the second block checks it).

In [ ]:
flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)  # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
n = len(mass)
w0, b0 = np.polyfit(flipper, mass, 1)       # the least-squares line
sigma = np.sqrt(np.mean((w0 * flipper + b0 - mass) ** 2))

def ln_lik(w, b, s):
    miss = mass - (b + w * (flipper - 198))
    return -n * np.log(s * np.sqrt(2 * np.pi)) - (miss ** 2).sum() / (2 * s ** 2)

print(round(sigma, 1))

*The lesson shows:*

```text
427.6
```

Now four lines, three from Module 00 and the best one, each with its mean squared error and its log-likelihood.

In [ ]:
for w, b in [(0, 4000), (30, 3940), (50, 3900), (w0, b0 + 198 * w0)]:
    mse = np.mean((mass - (b + w * (flipper - 198))) ** 2)
    print(round(float(w), 2), round(float(b), 2), round(mse, 1),
          round(ln_lik(w, b, sigma), 2))
b1 = b0 + 198 * w0
print([round(float(ln_lik(w0, b1, s)), 2) for s in (350, sigma, 500)])

*The lesson shows:*

```text
0.0 4000.0 564270.8 -102.24
30.0 3940.0 241945.8 -91.67
50.0 3900.0 183645.8 -89.75
48.91 3874.73 182842.1 -89.73
[-90.28, -89.73, -89.99]
```

## 8.5 Priors, and what weight decay assumes

Knowing something before the data is a prior, and a prior that prefers small weights is weight decay. This lesson explains a setting you will meet in Module 06 and again in Module 12.

[Read the lesson](https://aicourses.dev/mathematics/statistics/priors-and-weight-decay)

### What you believed before

This is last lesson’s log-likelihood with one more head and one more tail, so the same algebra gives the best setting $(h+1)/(h+t+2)$. For 3 heads in 3 flips that is $4/5 = 0.8$. For 7 in 10 it is $8/12 = 0.667$, pulled from 0.7 towards 0.5. For 700 in 1,000 it is $701/1002 = 0.6996$: with a lot of data the prior is washed out.

In [ ]:
import numpy as np

for heads, flips in [(3, 3), (7, 10), (700, 1000)]:
    print(heads, flips, heads / flips, round((heads + 1) / (flips + 2), 4))

*The lesson shows:*

```text
3 3 1.0 0.8
7 10 0.7 0.6667
700 1000 0.7 0.6996
```

### A prior on weights

For the twelve penguins, in the units of Module 04:

In [ ]:
flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)  # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
x = (flipper - flipper.mean()) / 10     # cm above average
y = (mass - mass.mean()) / 1000         # kg above average
sxx, sxy = (x * x).sum(), (x * y).sum()
print(round(sxx, 2), round(sxy, 3))
for lam in [0, sxx, 100]:
    print(round(lam, 2), round(sxy / (sxx + lam), 4))

*The lesson shows:*

```text
18.18 8.894
0 0.4891
18.18 0.2446
100 0.0753
```

At $\lambda = 0$ the weight is 0.4891 kilograms per centimetre (from the unrounded sums; the rounded ones above give 0.4892), Module 04’s answer. At $\lambda$ equal to the sum of squared inputs, 18.18, it halves. A check by brute force: search over many weights for the one with the lowest penalised loss.

In [ ]:
ws = np.linspace(0, 0.6, 601)
loss = [((w * x - y) ** 2).sum() + sxx * w ** 2 for w in ws]
print(round(ws[np.argmin(loss)], 3))

*The lesson shows:*

```text
0.245
```

## 8.6 Holding data out: overfitting

A wiggly curve through every point fits the past and misses the future, so you score on data the fit never saw. It is the most useful habit in the whole of machine learning.

[Read the lesson](https://aicourses.dev/mathematics/statistics/held-out-data)

### Scored on birds the fit never saw

In [ ]:
import numpy as np

flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)  # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
w, b = 48.91, 3875.0
miss = []
for f, m in [(181, 3750), (186, 3800), (195, 3250)]:
    miss.append(m - (b + w * (f - 198)))
print(np.round(miss, 1), round(np.mean(np.square(miss))))

*The lesson shows:*

```text
[ 706.5  511.9 -478.3] 329968
```

### The same fit on new birds

Now load the full table, take out the twelve, and fit polynomials of several degrees to the twelve only. A **polynomial of degree $d$** is a curve $c_0 + c_1 x + c_2 x^2 + \dots + c_d x^d$: degree 1 is the straight line, and each higher degree adds one more knob and one more bend. NumPy’s `np.polyfit` finds the least-squares knobs, and the code counts flippers from 198 in centimetres, so that the powers stay small.

In [ ]:
import urllib.request
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "penguins.csv", "penguins.csv")
F, M = np.loadtxt("penguins.csv", delimiter=",", skiprows=1,
                  usecols=(1, 2), unpack=True)  # the 342 birds
seen = np.zeros(len(F), dtype=bool)         # which of them are the twelve
for a, c in zip(flipper, mass):
    seen[np.where((F == a) & (M == c) & ~seen)[0][0]] = True
Fo, Mo = F[~seen], M[~seen]                 # 330 birds the fit never sees
print(seen.sum(), len(Fo))

*The lesson shows:*

```text
12 330
```

In [ ]:
def score(c, f, y):
    return np.mean((np.polyval(c, (f - 198) / 10) - y) ** 2)

for d in [1, 2, 4, 6]:
    c = np.polyfit((flipper - 198) / 10, mass, d)
    print(d, round(score(c, flipper, mass)), round(score(c, Fo, Mo)))

*The lesson shows:*

```text
1 182842 188891
2 144901 189408
4 85055 1070903
6 41854 35833622
```

Each row gives the degree, the training error and the held-out error. The training error falls with every extra knob, as it must, because each curve contains the simpler ones. The held-out error does not: degree 2 is no better than the line, degree 4 is nearly six times worse, and degree 6, whose training error is the lowest, misses held-out birds by a root mean square of about 5,990 g, more than a whole bird weighs. Part of that damage is birds outside the twelve’s range of 181 to 222 mm, where a high-degree curve swings wildly. The next lines keep only the held-out birds inside that range.

In [ ]:
inside = (Fo >= 181) & (Fo <= 222)
for d in [1, 6]:
    c = np.polyfit((flipper - 198) / 10, mass, d)
    print(d, inside.sum(), round(score(c, Fo[inside], Mo[inside])))

*The lesson shows:*

```text
1 293 183996
6 293 448196
```

## 8.7 How sure are we? Standard error and the bootstrap

An average of 20 wobbles more than an average of 2,000, and the bootstrap shows how much by resampling your own data. It gives you a range to put beside any number.

[Read the lesson](https://aicourses.dev/mathematics/statistics/how-sure)

### The wander has a name

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
n = len(mass)
print(round(mass.mean(), 1), round(mass.std(), 1),
      round(mass.std() / np.sqrt(n), 1))

*The lesson shows:*

```text
3862.5 738.5 213.2
```

### The bootstrap

In [ ]:
means = []
for _ in range(1000):
    pick = rng.integers(0, n, size=n)         # n positions, repeats allowed
    means.append(mass[pick].mean())
print(round(np.std(means), 1), np.percentile(means, [2.5, 97.5]).round(0))
means = mass[rng.integers(0, n, size=(1000, n))].mean(axis=1)
print(round(means.std(), 1))

*The lesson shows:*

```text
210.1 [3479. 4279.]
209.4
```

### A range, and what it says

The range says how far an average of twelve birds like these moves from one draw to the next. It does not say that the truth lies inside it with some chance, and it says nothing about single birds, which differ from each other by far more. Now the same method on the full table.

In [ ]:
import urllib.request
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "penguins.csv", "penguins.csv")
F, M = np.loadtxt("penguins.csv", delimiter=",", skiprows=1,
                  usecols=(1, 2), unpack=True)  # the 342 birds
big = M[rng.integers(0, len(M), size=(1000, len(M)))].mean(axis=1)
print(round(M.mean(), 1), round(M.std() / np.sqrt(len(M)), 1))
print(round(big.std(), 1), np.percentile(big, [2.5, 97.5]).round(0))

*The lesson shows:*

```text
4201.8 43.3
41.0 [4124. 4280.]
```

## 8.8 Regression to the mean

The best of many noisy runs looks worse next time with nothing changed. This lesson explains why, so you do not mistake luck for a result.

[Read the lesson](https://aicourses.dev/mathematics/statistics/regression-to-the-mean)

### The best of fifty

A score is made of two parts: what the model can do, its **skill**, and how the particular questions happened to fall, its **luck**. The luck wanders by about $\sqrt{p(1-p)/n}$ (the standard error of a share, from the last lesson), which for $p = 0.7$ and $n = 100$ questions is $\sqrt{0.21/100} = 0.046$: about 5 points. Taking the best of fifty scores picks the run with the biggest piece of good luck. The simulation below gives all fifty runs the same skill, so any difference is luck, and scores each run twice, on two fresh sets of 100 questions.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
first = rng.binomial(100, 0.70, size=50) / 100    # 50 runs, 100 questions
second = rng.binomial(100, 0.70, size=50) / 100   # the same runs, new questions
best = first.argmax()
print(first[best], second[best])
print(round(first.mean(), 3), round(second.mean(), 3))

*The lesson shows:*

```text
0.81 0.72
0.702 0.711
```

`rng.binomial(100, 0.70, size=50)` simulates 50 runs, each the number of right answers out of 100 when each is right with chance 0.70. The best run’s first score, 0.81, is far above 0.70 and its second, 0.72, is not. The two averages, over all fifty runs, are both close to 0.70. These are **simulated** numbers, with one seed. Repeat the whole experiment 2,000 times to see the typical case:

In [ ]:
firsts, seconds = [], []
for _ in range(2000):
    a = rng.binomial(100, 0.70, size=50) / 100
    b = rng.binomial(100, 0.70, size=50) / 100
    firsts.append(a.max())
    seconds.append(b[a.argmax()])
print(round(np.mean(firsts), 3), round(np.mean(seconds), 3))

*The lesson shows:*

```text
0.8 0.7
```

### How much it slides back

In [ ]:
skill = rng.normal(0.70, 0.02, size=(2000, 50))
a = rng.binomial(100, skill) / 100
b = rng.binomial(100, skill) / 100
top = a.argmax(axis=1)
rows = np.arange(2000)
gain1 = (a[rows, top] - 0.70).mean()
gain2 = (b[rows, top] - 0.70).mean()
print(round(gain1, 3), round(gain2, 3), round(gain2 / gain1, 2))

*The lesson shows:*

```text
0.109 0.016 0.15
```

## 8.9 Project: fit a noisy line three ways

You fit the full penguin table three ways, check it on held-out data and write what the fit can and cannot claim. It joins every lesson of the module into one defensible result.

[Read the lesson](https://aicourses.dev/mathematics/statistics/project-fit-a-noisy-line)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Step 1: least squares

Load the table and fit the line. The line that fits all 342 birds best has $w = 49.6856$ grams per millimetre and, with flipper length counted from zero, $b = -5780.83$ g, with a mean squared error of 154,546.2 g² (a typical miss of 393.1 g). In the form of Module 00, counted from 198 mm, the line is $b + w(\text{flipper} - 198)$ with $b = -5780.83 + 198 \times 49.6856 = 4056.92$ g (4056.91 from the unrounded values, as the code prints): the predicted mass of a 198 mm bird.

In [ ]:
import numpy as np, urllib.request

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "penguins.csv", "penguins.csv")
F, M = np.loadtxt("penguins.csv", delimiter=",", skiprows=1,
                  usecols=(1, 2), unpack=True)
w, b = np.polyfit(F, M, 1)
mse = np.mean((w * F + b - M) ** 2)
print(len(F), round(w, 4), round(b, 2), round(mse, 1), round(mse ** 0.5, 1))
print("course form: b =", round(b + 198 * w, 2))

*The lesson shows:*

```text
342 49.6856 -5780.83 154546.2 393.1
course form: b = 4056.91
```

### Step 2: the same with weight decay

Now the penalty of lesson 5, on the one-knob form of Module 04: flipper in centimetres and mass in kilograms, both counted from their averages, so that the best weight is $\sum xy / (\sum xx + \lambda)$.

In [ ]:
x = (F - F.mean()) / 10          # cm above average
y = (M - M.mean()) / 1000        # kg above average
sxx, sxy = (x * x).sum(), (x * y).sum()
for lam in [0, 10, 100, 1000]:
    wl = sxy / (sxx + lam)
    print(lam, round(wl, 4), round(np.mean((wl * x - y) ** 2) * 1e6))

*The lesson shows:*

```text
0 0.4969 154546
10 0.4896 154650
100 0.4327 162665
1000 0.2001 328173
```

### Step 3: held out, and a range

Hold out 100 birds with a seeded shuffle, fit on the other 242, and score on the 100. The centring must use the training birds only: centre on their averages, not on all 342.

In [ ]:
order = rng.permutation(len(F))
tr, te = order[:242], order[242:]
x = (F - F[tr].mean()) / 10
y = (M - M[tr].mean()) / 1000
sxx, sxy = (x[tr] ** 2).sum(), (x[tr] * y[tr]).sum()
for lam in [0, 10, 100, 1000]:
    wl = sxy / (sxx + lam)
    print(lam, round(wl, 4), round(np.mean((wl * x[te] - y[te]) ** 2) * 1e6))

*The lesson shows:*

```text
0 0.4915 142690
10 0.482 143586
100 0.4106 160074
1000 0.1654 347520
```

The held-out error is lowest with no penalty, and rises with it. For this fit, weight decay is not needed: one knob and 242 birds leave little room to chase noise. Now a range for the held-out error of the lambda-zero fit, and one for the slope, both by the bootstrap of lesson 7.

In [ ]:
sq = (sxy / sxx * x[te] - y[te]) ** 2 * 1e6       # held-out squared misses
boot = sq[rng.integers(0, 100, size=(1000, 100))].mean(axis=1)
print(round(sq.mean()), np.percentile(boot, [2.5, 97.5]).round(0))
slopes = []
for _ in range(1000):
    pick = tr[rng.integers(0, 242, size=242)]      # resample training birds
    slopes.append(np.polyfit(F[pick], M[pick], 1)[0])
print(round(np.mean(slopes), 2), np.percentile(slopes, [2.5, 97.5]).round(2))

*The lesson shows:*

```text
142690 [108855. 178123.]
49.13 [45.83 52.13]
```

### What the fit can and cannot claim

Before you write, run one more check on the groups of lesson 2.

In [ ]:
sp = np.loadtxt("penguins.csv", delimiter=",", skiprows=1,
                usecols=0, dtype=str)
for s in ["Adelie", "Chinstrap", "Gentoo"]:
    k = sp == s
    print(s, k.sum(), round(np.polyfit(F[k], M[k], 1)[0], 1))

*The lesson shows:*

```text
Adelie 151 32.8
Chinstrap 68 34.6
Gentoo 123 54.6
```